In [44]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings("ignore")

SEED = 65


In [45]:
train = pd.read_csv("/kaggle/input/playground-series-s5e11/train.csv", index_col="id")
test  = pd.read_csv("/kaggle/input/playground-series-s5e11/test.csv", index_col="id")

sample_sub = pd.read_csv("/kaggle/input/playground-series-s5e11/sample_submission.csv")

# Optional external dataset (but we keep off by default)
# external = pd.read_csv("/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv")
# external = external[train.columns.tolist()]


In [46]:
cat_cols = train.select_dtypes(exclude="number").columns.tolist()
num_cols = train.select_dtypes(include="number").columns.tolist()
target = "loan_paid_back"

num_cols.remove(target)


# 4. Count Encoding Preprocessing

In [47]:
def preprocessor_count(df, cat_feats, num_feats, reference_df):
    df = df.copy()
    ref_df = reference_df.copy()

    for cat in cat_feats:
        freq = ref_df[cat].value_counts() / len(ref_df)
        df[f"{cat}_count"] = df[cat].map(freq)
        df = df.drop(columns=[cat])

    return df


In [50]:
train_proc = preprocessor_count(train, cat_cols, num_cols, reference_df=train)
test_proc  = preprocessor_count(test,  cat_cols, num_cols, reference_df=train)

# Building the train matrices
X = train_proc.drop(columns=[target])
y = train_proc[target]

X_test = test_proc.copy()
test_ids = test_proc.index


## 5. APply preprocessing

In [51]:
train_proc = preprocessor_count(train, cat_cols, num_cols, reference_df=train)
test_proc  = preprocessor_count(test,  cat_cols, num_cols, reference_df=train)

X = train_proc.drop(columns=[target])
y = train_proc[target]

X_test = test_proc.copy()


In [38]:
model = CatBoostClassifier(
    iterations=30000,
    learning_rate=0.024,
    depth=3,
    l2_leaf_reg=5,
    random_strength=2.5,
    bagging_temperature=5.0,
    border_count=450,
    grow_policy='Depthwise',
    boosting_type='Plain',
    eval_metric='AUC',
    early_stopping_rounds=500,
    eval_fraction=0.2,
    verbose=500,
    random_seed=SEED,
    use_best_model=True,
    od_type='Iter'
)
# Hyperparams found in the notebook
model.fit(X, y)


0:	test: 0.8434534	best: 0.8434534 (0)	total: 122ms	remaining: 1h 1m 13s
500:	test: 0.9162857	best: 0.9162857 (500)	total: 31.6s	remaining: 30m 58s
1000:	test: 0.9195233	best: 0.9195236 (999)	total: 1m 2s	remaining: 30m 5s
1500:	test: 0.9216779	best: 0.9216779 (1500)	total: 1m 33s	remaining: 29m 28s
2000:	test: 0.9227337	best: 0.9227337 (2000)	total: 2m 3s	remaining: 28m 52s
2500:	test: 0.9236382	best: 0.9236382 (2500)	total: 2m 34s	remaining: 28m 17s
3000:	test: 0.9243039	best: 0.9243039 (3000)	total: 3m 5s	remaining: 27m 45s
3500:	test: 0.9248009	best: 0.9248013 (3496)	total: 3m 35s	remaining: 27m 13s
4000:	test: 0.9251301	best: 0.9251301 (4000)	total: 4m 6s	remaining: 26m 43s
4500:	test: 0.9254608	best: 0.9254612 (4494)	total: 4m 37s	remaining: 26m 12s
5000:	test: 0.9257175	best: 0.9257182 (4994)	total: 5m 8s	remaining: 25m 40s
5500:	test: 0.9259152	best: 0.9259155 (5499)	total: 5m 38s	remaining: 25m 9s
6000:	test: 0.9260380	best: 0.9260380 (6000)	total: 6m 9s	remaining: 24m 38s
650

In [ ]:
model.save_model('/kaggle/working/saved_models/catboost_model.cbm')

DONT TOUCH THE ABOVE CELL PLS

In [52]:
ts_proba = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": test.index,
    "loan_paid_back": ts_proba
})

submission.to_csv("submission2.csv", index=False)


In [43]:
tr_01 = preprocessor_count(train)
ts_01 = preprocessor_count(test)



TypeError: preprocessor_count() missing 3 required positional arguments: 'cat_feats', 'num_feats', and 'reference_df'